# Module 09: RL for Object Detection
## Notebook 5: Visual Question Answering with RL Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_09_RL_For_Object_Detection/05_Visual_Question_Answering/visual_question_answering.ipynb)

---

### Learning Objectives

1. Formulate attention-based VQA as an MDP
2. Design state representations combining image features and question embeddings
3. Implement an RL agent that learns which image regions to attend for different questions
4. Visualize attention maps for different question types
5. Analyze correlation between question semantics and attended regions

---

**Key Idea:** Given an image and a natural language question, the RL agent learns to select the most informative image region to look at before answering. This is analogous to how humans focus on different parts of an image depending on what is being asked.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for object detection (TINY - under 200MB)
import torchvision
import numpy as np

# MNIST: Real handwritten digits for detection scenes
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST: {len(mnist)} real handwritten digits (11MB)")

# CIFAR-10: Real photographs
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB)")

# Create detection dataset from MNIST (place digits on canvas with known bounding boxes)
def create_detection_dataset(n_scenes=200, canvas_size=84, max_objects=3):
    """Create real detection scenes using MNIST digits on canvas."""
    scenes = []
    for _ in range(n_scenes):
        canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        boxes, labels = [], []
        n_obj = np.random.randint(1, max_objects + 1)
        for _ in range(n_obj):
            idx = np.random.randint(len(mnist))
            digit_img = np.array(mnist[idx][0])
            label = mnist[idx][1]
            x = np.random.randint(0, canvas_size - 28)
            y = np.random.randint(0, canvas_size - 28)
            canvas[y:y+28, x:x+28] = np.maximum(canvas[y:y+28, x:x+28], digit_img)
            boxes.append([x, y, x+28, y+28])
            labels.append(label)
        scenes.append({'image': canvas, 'boxes': boxes, 'labels': labels})
    return scenes

detection_data = create_detection_dataset(200)
print(f"✅ Created {len(detection_data)} detection scenes with REAL digit images")
print(f"   Each scene has 1-3 objects with ground-truth bounding boxes")
print(f"   Total download: ~181MB (MNIST + CIFAR-10)")

## Deep Derivation: Visual Question Answering with RL-Guided Attention

### Step 1: VQA as Conditional Distribution
Given image $I$ and question $q$, VQA predicts answer $a$:
$$p(a | I, q) = \text{softmax}(W_a \cdot f(v, q) + b_a)$$

where $v = \text{CNN}(I) \in \mathbb{R}^{K \times d_v}$ are spatial features ($K$ regions, $d_v$ feature dim), and $q$ is encoded by an RNN/Transformer.

**Cross-modal fusion function $f$:**
$$f(v, q) = \sum_{k=1}^K \alpha_k \cdot v_k, \quad \alpha_k = \text{softmax}(w^T \tanh(W_v v_k + W_q q))$$

This is the attention mechanism — the question "attends" to relevant image regions.

### Step 2: Bilinear Attention Networks (Detailed Derivation)
Simple dot-product attention limits interaction. Bilinear attention computes:
$$\alpha_{ij} = v_i^T W q_j$$

where $W \in \mathbb{R}^{d_v \times d_q}$. But $W$ has $d_v \times d_q$ parameters (potentially millions).

**Low-rank factorization:** $W = U V^T$ where $U \in \mathbb{R}^{d_v \times r}$, $V \in \mathbb{R}^{d_q \times r}$:
$$\alpha_{ij} = v_i^T U V^T q_j = (U^T v_i)^T (V^T q_j)$$

**Derivation:** This reduces parameters from $d_v \cdot d_q$ to $(d_v + d_q) \cdot r$. For $d_v = d_q = 2048$ and $r = 64$:
- Full: $2048 \times 2048 = 4,194,304$ parameters
- Low-rank: $(2048 + 2048) \times 64 = 262,144$ parameters (16× reduction) $\blacksquare$

### Step 3: RL for Sequential Visual Reasoning
Complex questions require multi-step reasoning. Formulate as MDP:

**State:** $s_t = (q, v, m_t)$ where $m_t$ is a memory vector accumulating evidence.
**Action:** Select which region to examine: $a_t \in \{1, \ldots, K\}$
**Transition:** $m_{t+1} = \text{GRU}(m_t, v_{a_t}, q)$
**Policy:** $\pi_\theta(a_t | s_t) = \text{softmax}(W_\pi [m_t; q])$

**Reward:** Only at final step $T$:
$$R_T = \begin{cases} +1 & \text{if predicted answer is correct} \\ -1 & \text{otherwise} \end{cases}$$

This is a sparse reward problem! The agent must learn which regions to attend to over multiple steps.

### Step 4: REINFORCE with Self-Critical Baseline
Since rewards are sparse, variance reduction is critical. Use the self-critical baseline (Rennie et al., 2017):
$$b = R(\hat{a}^{\text{greedy}})$$

where $\hat{a}^{\text{greedy}}$ is the answer obtained by greedily selecting attention (argmax at each step).

**Gradient estimator:**
$$\nabla_\theta J = E_{\pi_\theta}\left[\left(R(a^{\text{sample}}) - R(\hat{a}^{\text{greedy}})\right) \nabla_\theta \log \pi_\theta(a^{\text{sample}})\right]$$

**Proof that this reduces variance:** Let $\hat{g}$ be the gradient estimate. With baseline $b$:
$$\text{Var}(\hat{g}_b) = E[(R - b)^2 (\nabla\log\pi)^2] - (E[(R-b)\nabla\log\pi])^2$$

Since $E[(R-b)\nabla\log\pi] = E[R \cdot \nabla\log\pi]$ (baseline doesn't change expectation), the variance becomes:
$$\text{Var}(\hat{g}_b) = E[R^2(\nabla\log\pi)^2] - 2bE[R(\nabla\log\pi)^2] + b^2 E[(\nabla\log\pi)^2] - (E[R\nabla\log\pi])^2$$

Minimizing w.r.t. $b$: $\frac{\partial}{\partial b}\text{Var} = -2E[R(\nabla\log\pi)^2] + 2b E[(\nabla\log\pi)^2] = 0$

$$b^* = \frac{E[R(\nabla\log\pi)^2]}{E[(\nabla\log\pi)^2]} \quad \blacksquare$$

### Step 5: Cross-Entropy Loss for Answer Classification
With $N$ candidate answers:
$$\mathcal{L}_{\text{CE}} = -\sum_{i=1}^N y_i \log p(a_i | I, q)$$

For soft labels (multiple valid answers with confidence scores $c_i$):
$$y_i = \min\left(\frac{c_i}{3}, 1\right)$$

**Derivation of soft label formula:** In VQA v2, each question has 10 human annotations. If $c_i$ annotators give answer $a_i$:
$$y_i = \min\left(\frac{c_i}{3}, 1\right)$$

This means an answer needs at least 3 annotators to get full credit. The $\min$ caps at 1 to maintain a valid distribution (after renormalization).

**VQA accuracy metric:**
$$\text{Acc}(a) = \min\left(\frac{\#\text{humans who said } a}{3}, 1\right)$$

### Step 6: Multi-Modal Transformer (Formal Attention)
In a multi-modal Transformer, cross-attention between modalities:
$$\text{CrossAttn}(Q_q, K_v, V_v) = \text{softmax}\left(\frac{Q_q K_v^T}{\sqrt{d}}\right) V_v$$

where $Q_q = W_Q \cdot H_q$ (query from question), $K_v = W_K \cdot H_v$, $V_v = W_V \cdot H_v$ (key/value from image).

**Self-attention computational cost:**
$$O(\text{self-attn}) = O(n^2 \cdot d)$$

For $n = K + L$ (K image regions + L question tokens), this is:
$$O((K+L)^2 \cdot d) = O(K^2 d + 2KLd + L^2 d)$$

The cross-modal term $O(KLd)$ enables question-image interaction.

### Step 7: Information Gain from Attended Regions
Define the information gain of attending to region $k$:
$$\text{IG}(k) = H(A | q, m_t) - H(A | q, m_t, v_k)$$

where $A$ is the answer random variable. The optimal policy attends to regions maximizing expected IG:
$$a_t^* = \arg\max_k E[\text{IG}(k) | s_t]$$

**Proof that IG-maximizing policy minimizes expected steps:** By the submodularity of mutual information, greedily selecting the highest-IG region at each step achieves at least $(1 - 1/e)$ of the optimal batch information gain. This gives an approximation guarantee:
$$\sum_{t=1}^T \text{IG}(a_t^*) \geq \left(1 - \frac{1}{e}\right) \max_{|S|=T} \text{IG}(S) \quad \blacksquare$$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import deque, Counter
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

## 1. Mathematical Formulation: MDP for Attention-Based VQA

### Problem Setting

Given an image $\mathbf{I}$ and a question $q$, the agent must select an image region to attend to, then produce an answer based on the attended region's features combined with the question.

### MDP Definition $(\mathcal{S}, \mathcal{A}, \mathcal{T}, \mathcal{R}, \gamma)$

### State Space $\mathcal{S}$

$$s = \left(\mathbf{F},\; \mathbf{q}\right)$$

where:
- $\mathbf{F} = \{\mathbf{f}_1, \ldots, \mathbf{f}_{N^2}\} \in \mathbb{R}^{N^2 \times d_f}$ is the spatial feature grid
  - $\mathbf{f}_{ij}$ = features of region $(i,j)$ in an $N \times N$ grid
- $\mathbf{q} \in \mathbb{R}^{d_q}$ is the question embedding

**Question Embedding (Bag-of-Words):**

$$\mathbf{q} = \frac{1}{|w|} \sum_{w_k \in \text{question}} \mathbf{e}(w_k)$$

where $\mathbf{e}(w_k)$ is the one-hot or learned embedding of word $w_k$.

### Action Space $\mathcal{A}$

The agent selects one of $N \times N$ image regions to attend:

$$\mathcal{A} = \{(i,j) : i \in \{0,\ldots,N-1\}, j \in \{0,\ldots,N-1\}\}$$

This defines a spatial attention map:

$$\alpha_{ij} = \begin{cases} 1 & \text{if } (i,j) = a \\ 0 & \text{otherwise} \end{cases}$$

(Hard attention — the agent commits to one region.)

### Answer Generation

After attending to region $(i,j)$, the answer is computed:

$$\hat{y} = g(\mathbf{f}_{ij}, \mathbf{q}) = \arg\max_c \; \mathbf{W}_c^\top [\mathbf{f}_{ij}; \mathbf{q}] + b_c$$

where $[;]$ denotes concatenation and $c$ ranges over possible answers.

### Reward Function $\mathcal{R}$

$$R(s, a) = \begin{cases}
+1 & \text{if } \hat{y}(\mathbf{f}_a, \mathbf{q}) = y^* \text{ (correct answer)} \\
0 & \text{otherwise}
\end{cases}$$

### Policy Optimization

Since this is a single-step decision (attend then answer), we optimize:

$$\pi^* = \arg\max_\pi \mathbb{E}_{(\mathbf{I}, q, y^*)}\left[R\left(s, \pi(s)\right)\right]$$

This is equivalent to maximizing VQA accuracy by learning optimal attention.

### Q-Function

For this single-step setting:

$$Q(s, a) = \mathbb{E}[R(s, a)] = \mathbb{P}[\text{correct answer} \mid \text{attend to region } a, \text{question } q]$$

The Q-network learns to predict which region leads to a correct answer for each question type.

## 2. Synthetic VQA Dataset

We create synthetic images with colored shapes at known positions, with simple questions about:
- Color: "What color is the object on the [left/right/top/bottom]?"
- Count: "How many [circles/squares] are there?"
- Position: "Where is the [red/blue/green] object?"

In [ ]:
class VQAEnvironment:
    """Simple VQA environment with synthetic images and questions."""
    
    def __init__(self, image_size=64, grid_size=4):
        self.image_size = image_size
        self.grid_size = grid_size
        self.cell_size = image_size // grid_size
        self.n_actions = grid_size * grid_size
        
        # Vocabulary
        self.colors = ['red', 'blue', 'green', 'yellow']
        self.shapes = ['circle', 'square']
        self.positions = ['left', 'right', 'top', 'bottom']
        
        # Build vocabulary for bag-of-words
        self.vocab = ['what', 'color', 'is', 'the', 'object', 'on', 'how',
                      'many', 'are', 'there', 'where', 'circles', 'squares',
                      'left', 'right', 'top', 'bottom', 'red', 'blue',
                      'green', 'yellow', '?']
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        self.vocab_size = len(self.vocab)
        
        # Possible answers
        self.answers = self.colors + ['1', '2', '3', '4'] + self.positions
        self.answer2idx = {a: i for i, a in enumerate(self.answers)}
        self.n_answers = len(self.answers)
        
        # Color RGB values
        self.color_rgb = {
            'red': [0.9, 0.2, 0.2],
            'blue': [0.2, 0.2, 0.9],
            'green': [0.2, 0.8, 0.2],
            'yellow': [0.9, 0.9, 0.2]
        }
    
    def _encode_question(self, question):
        """Bag-of-words encoding of question."""
        words = question.lower().replace('?', ' ?').split()
        encoding = np.zeros(self.vocab_size, dtype=np.float32)
        for w in words:
            if w in self.word2idx:
                encoding[self.word2idx[w]] = 1.0
        return encoding / max(np.sum(encoding), 1.0)
    
    def _generate_scene(self):
        """Generate a scene with colored shapes."""
        image = np.ones((self.image_size, self.image_size, 3)) * 0.15
        
        # Place 2-4 objects
        n_objects = np.random.randint(2, 5)
        self.scene_objects = []
        
        # Available grid positions
        positions = [(i, j) for i in range(self.grid_size) for j in range(self.grid_size)]
        chosen_pos = random.sample(positions, min(n_objects, len(positions)))
        
        for (gi, gj) in chosen_pos:
            color = random.choice(self.colors)
            shape = random.choice(self.shapes)
            
            # Center of the grid cell
            cx = gj * self.cell_size + self.cell_size // 2
            cy = gi * self.cell_size + self.cell_size // 2
            size = self.cell_size // 2 - 2
            
            rgb = self.color_rgb[color]
            
            if shape == 'circle':
                y, x = np.ogrid[:self.image_size, :self.image_size]
                mask = (x - cx)**2 + (y - cy)**2 <= size**2
                image[mask] = rgb
            else:  # square
                y1, y2 = cy - size, cy + size
                x1, x2 = cx - size, cx + size
                y1, y2 = max(0, y1), min(self.image_size, y2)
                x1, x2 = max(0, x1), min(self.image_size, x2)
                image[y1:y2, x1:x2] = rgb
            
            self.scene_objects.append({
                'color': color,
                'shape': shape,
                'grid_pos': (gi, gj),
                'pixel_pos': (cx, cy)
            })
        
        return image
    
    def _generate_question(self):
        """Generate a question and its correct answer."""
        q_type = np.random.choice(['color', 'count', 'position'])
        
        if q_type == 'color':
            # "What color is the object on the [position]?"
            pos = random.choice(self.positions)
            question = f"What color is the object on the {pos}?"
            
            # Find object closest to that position
            target_obj = self._find_object_at_position(pos)
            if target_obj is not None:
                answer = target_obj['color']
                target_region = target_obj['grid_pos']
            else:
                answer = random.choice(self.colors)
                target_region = (0, 0)
                
        elif q_type == 'count':
            # "How many [shape]s are there?"
            shape = random.choice(self.shapes)
            question = f"How many {shape}s are there?"
            
            count = sum(1 for obj in self.scene_objects if obj['shape'] == shape)
            answer = str(min(count, 4))
            # For count, the relevant regions are all objects of that shape
            shape_objs = [obj for obj in self.scene_objects if obj['shape'] == shape]
            if shape_objs:
                target_region = shape_objs[0]['grid_pos']
            else:
                target_region = (self.grid_size//2, self.grid_size//2)
                
        else:  # position
            # "Where is the [color] object?"
            color_objs = [obj for obj in self.scene_objects]
            if color_objs:
                target_obj = random.choice(color_objs)
                question = f"Where is the {target_obj['color']} object?"
                
                # Determine position label
                gi, gj = target_obj['grid_pos']
                if gj < self.grid_size // 2:
                    answer = 'left'
                elif gj >= self.grid_size // 2:
                    answer = 'right'
                if gi < self.grid_size // 2:
                    answer = 'top'
                elif gi >= self.grid_size // 2:
                    answer = 'bottom'
                target_region = target_obj['grid_pos']
            else:
                question = "Where is the red object?"
                answer = 'left'
                target_region = (0, 0)
        
        return question, answer, target_region, q_type
    
    def _find_object_at_position(self, position):
        """Find the object closest to the specified position."""
        if not self.scene_objects:
            return None
        
        best_obj = None
        best_score = -float('inf')
        
        for obj in self.scene_objects:
            gi, gj = obj['grid_pos']
            if position == 'left':
                score = -gj
            elif position == 'right':
                score = gj
            elif position == 'top':
                score = -gi
            else:  # bottom
                score = gi
            
            if score > best_score:
                best_score = score
                best_obj = obj
        
        return best_obj
    
    def _get_region_features(self):
        """Extract features for each grid region."""
        features = []
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                y1 = i * self.cell_size
                y2 = (i + 1) * self.cell_size
                x1 = j * self.cell_size
                x2 = (j + 1) * self.cell_size
                
                region = self.image[y1:y2, x1:x2]
                # Features: mean RGB + std RGB + max RGB + has_object
                feat = np.concatenate([
                    region.mean(axis=(0, 1)),  # mean RGB
                    region.std(axis=(0, 1)),   # std RGB
                    region.max(axis=(0, 1)),   # max RGB
                    [1.0 if region.mean() > 0.3 else 0.0]  # has content
                ])
                features.append(feat)
        
        return np.array(features, dtype=np.float32)  # (N*N, 10)
    
    def reset(self):
        """Generate new scene and question."""
        self.image = self._generate_scene()
        self.question, self.answer, self.target_region, self.q_type = self._generate_question()
        self.question_encoding = self._encode_question(self.question)
        self.region_features = self._get_region_features()
        
        # State: flattened region features + question encoding
        state = np.concatenate([
            self.region_features.flatten(),
            self.question_encoding
        ])
        
        return state
    
    def step(self, action):
        """Agent attends to a region and tries to answer."""
        # Get features of attended region
        attended_features = self.region_features[action]
        
        # Simple rule-based answer based on attended region
        attended_row = action // self.grid_size
        attended_col = action % self.grid_size
        
        # Check if the attended region is the correct target
        correct_action = self.target_region[0] * self.grid_size + self.target_region[1]
        
        # Reward: +1 if attended to the right region
        if action == correct_action:
            reward = 1.0
        elif abs(attended_row - self.target_region[0]) + abs(attended_col - self.target_region[1]) <= 1:
            reward = 0.3  # Close
        else:
            reward = 0.0
        
        info = {
            'question': self.question,
            'answer': self.answer,
            'q_type': self.q_type,
            'target_region': self.target_region,
            'attended_region': (attended_row, attended_col),
            'correct': action == correct_action
        }
        
        return None, reward, True, info  # Single step episode

# Test
env = VQAEnvironment()
state = env.reset()
print(f"State dim: {state.shape[0]}")
print(f"  Region features: {env.grid_size}x{env.grid_size} x 10 = {env.grid_size**2 * 10}")
print(f"  Question encoding: {env.vocab_size}")
print(f"Actions (regions): {env.n_actions} ({env.grid_size}x{env.grid_size})")
print(f"\nSample question: '{env.question}'")
print(f"Answer: '{env.answer}'")
print(f"Target region: {env.target_region}")
print(f"Objects in scene: {len(env.scene_objects)}")

## 3. Visualize Sample Scenes and Questions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx in range(8):
    ax = axes[idx // 4, idx % 4]
    state = env.reset()
    
    ax.imshow(env.image)
    
    # Draw grid
    for i in range(env.grid_size + 1):
        ax.axhline(y=i * env.cell_size, color='white', linewidth=0.5, alpha=0.3)
        ax.axvline(x=i * env.cell_size, color='white', linewidth=0.5, alpha=0.3)
    
    # Highlight target region
    tr, tc = env.target_region
    rect = patches.Rectangle(
        (tc * env.cell_size, tr * env.cell_size),
        env.cell_size, env.cell_size,
        linewidth=2, edgecolor='white', facecolor='white', alpha=0.2
    )
    ax.add_patch(rect)
    
    ax.set_title(f'Q: {env.question}\nA: {env.answer} [{env.q_type}]', fontsize=8)
    ax.axis('off')

plt.suptitle('Sample VQA Scenes (white highlight = correct attention region)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Q-Network for VQA Attention

In [ ]:
class VQAAttentionNet(nn.Module):
    """Q-network that learns which region to attend given image + question."""
    
    def __init__(self, state_dim, n_actions):
        super(VQAAttentionNet, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )
    
    def forward(self, x):
        return self.net(x)


class VQAAgent:
    def __init__(self, state_dim, n_actions, lr=1e-3,
                 eps_start=1.0, eps_end=0.05, eps_decay=0.997):
        self.n_actions = n_actions
        self.epsilon = eps_start
        self.eps_end = eps_end
        self.eps_decay = eps_decay
        
        self.q_net = VQAAttentionNet(state_dim, n_actions).to(device)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = deque(maxlen=5000)
        self.batch_size = 32
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_net(state_t)
            return q_values.argmax(1).item()
    
    def get_attention_map(self, state):
        """Get soft attention map (Q-values as attention weights)."""
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_net(state_t).squeeze(0).cpu().numpy()
            # Softmax to get attention weights
            exp_q = np.exp(q_values - q_values.max())
            attention = exp_q / exp_q.sum()
            return attention
    
    def store(self, state, action, reward):
        self.buffer.append((state, action, reward))
    
    def update(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        
        batch = random.sample(self.buffer, self.batch_size)
        states, actions, rewards = zip(*batch)
        
        states_t = torch.FloatTensor(np.array(states)).to(device)
        actions_t = torch.LongTensor(actions).to(device)
        rewards_t = torch.FloatTensor(rewards).to(device)
        
        # For single-step: Q(s,a) should predict expected reward
        q_values = self.q_net(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)
        
        loss = nn.MSELoss()(q_values, rewards_t)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        self.epsilon = max(self.eps_end, self.epsilon * self.eps_decay)
        return loss.item()

state_dim = env.grid_size**2 * 10 + env.vocab_size
agent = VQAAgent(state_dim=state_dim, n_actions=env.n_actions)
print(f"State dim: {state_dim}")
print(agent.q_net)

## 5. Training

In [ ]:
def train_vqa_agent(agent, env, n_episodes=2000):
    """Train the VQA attention agent."""
    correct_hist = []
    reward_hist = []
    loss_hist = []
    type_correct = {'color': [], 'count': [], 'position': []}
    
    for ep in range(n_episodes):
        state = env.reset()
        action = agent.select_action(state)
        _, reward, _, info = env.step(action)
        
        agent.store(state, action, reward)
        loss = agent.update()
        
        correct_hist.append(info['correct'])
        reward_hist.append(reward)
        type_correct[info['q_type']].append(info['correct'])
        if loss > 0:
            loss_hist.append(loss)
        
        if (ep + 1) % 300 == 0:
            acc = np.mean(correct_hist[-300:])
            print(f"Episode {ep+1:5d} | Accuracy: {acc:.2%} | "
                  f"Epsilon: {agent.epsilon:.3f}")
            for qt in type_correct:
                recent = type_correct[qt][-100:]
                if recent:
                    print(f"  {qt:10s}: {np.mean(recent):.2%}")
    
    return correct_hist, reward_hist, loss_hist, type_correct

correct_hist, reward_hist, loss_hist, type_correct = train_vqa_agent(agent, env, n_episodes=2000)

## 6. Training Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

window = 100

# Overall accuracy
acc_smooth = [np.mean(correct_hist[max(0,i-window):i+1]) for i in range(len(correct_hist))]
axes[0, 0].plot(acc_smooth, color='blue', linewidth=1.5)
axes[0, 0].axhline(y=1.0/env.n_actions, color='red', linestyle='--', 
                   alpha=0.5, label=f'Random ({1/env.n_actions:.1%})')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Attention Accuracy (rolling window)')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1.0)
axes[0, 0].grid(True, alpha=0.3)

# By question type
for qt, color in zip(['color', 'count', 'position'], ['red', 'blue', 'green']):
    data = type_correct[qt]
    if len(data) > window:
        smoothed = [np.mean(data[max(0,i-window):i+1]) for i in range(len(data))]
        axes[0, 1].plot(smoothed, label=qt, color=color, linewidth=1.5)
axes[0, 1].set_xlabel('Episode (per type)')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Accuracy by Question Type')
axes[0, 1].legend()
axes[0, 1].set_ylim(0, 1.0)
axes[0, 1].grid(True, alpha=0.3)

# Loss
if loss_hist:
    loss_smooth = np.convolve(loss_hist, np.ones(50)/50, mode='valid')
    axes[1, 0].plot(loss_smooth, color='orange', linewidth=1.0)
    axes[1, 0].set_xlabel('Training Step')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title('Training Loss')
    axes[1, 0].grid(True, alpha=0.3)

# Final accuracy breakdown
final_window = 300
final_accs = {}
for qt in type_correct:
    data = type_correct[qt][-final_window:]
    if data:
        final_accs[qt] = np.mean(data)
final_accs['overall'] = np.mean(correct_hist[-final_window:])

colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
bars = axes[1, 1].bar(final_accs.keys(), final_accs.values(), 
                      color=colors_bar, edgecolor='black')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title(f'Final Accuracy (last {final_window} episodes)')
axes[1, 1].set_ylim(0, 1.0)
axes[1, 1].axhline(y=1/env.n_actions, color='gray', linestyle='--', alpha=0.5)
for i, (k, v) in enumerate(final_accs.items()):
    axes[1, 1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold', fontsize=9)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('vqa_training.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Attention Map Visualization

In [ ]:
def visualize_attention_maps(agent, env, n_examples=6):
    """Visualize where the agent attends for different questions."""
    fig, axes = plt.subplots(3, n_examples, figsize=(3.5*n_examples, 10))
    
    for idx in range(n_examples):
        state = env.reset()
        attention = agent.get_attention_map(state)
        attention_grid = attention.reshape(env.grid_size, env.grid_size)
        
        action = agent.select_action(state, training=False)
        _, reward, _, info = env.step(action)
        
        # Image
        axes[0, idx].imshow(env.image)
        axes[0, idx].set_title(f'Q: {env.question}', fontsize=8, wrap=True)
        axes[0, idx].axis('off')
        
        # Attention heatmap overlay
        axes[1, idx].imshow(env.image)
        # Upsample attention to image size
        attn_upsampled = np.kron(attention_grid, np.ones((env.cell_size, env.cell_size)))
        axes[1, idx].imshow(attn_upsampled, cmap='hot', alpha=0.5, vmin=0)
        
        # Mark selected region
        sel_row = action // env.grid_size
        sel_col = action % env.grid_size
        rect = patches.Rectangle(
            (sel_col * env.cell_size, sel_row * env.cell_size),
            env.cell_size, env.cell_size,
            linewidth=2, edgecolor='cyan', facecolor='none'
        )
        axes[1, idx].add_patch(rect)
        
        status = '✓' if info['correct'] else '✗'
        axes[1, idx].set_title(f'Attention {status}\nA: {env.answer}', fontsize=9)
        axes[1, idx].axis('off')
        
        # Attention grid values
        im = axes[2, idx].imshow(attention_grid, cmap='YlOrRd', vmin=0)
        for i in range(env.grid_size):
            for j in range(env.grid_size):
                val = attention_grid[i, j]
                axes[2, idx].text(j, i, f'{val:.2f}', ha='center', va='center',
                                 fontsize=7, color='black' if val < 0.3 else 'white')
        axes[2, idx].set_title(f'Attention Weights [{info["q_type"]}]', fontsize=9)
        axes[2, idx].axis('off')
    
    axes[0, 0].set_ylabel('Image', fontsize=11)
    axes[1, 0].set_ylabel('Attention Overlay', fontsize=11)
    axes[2, 0].set_ylabel('Attention Grid', fontsize=11)
    
    plt.suptitle('VQA Attention Maps: Where the Agent Looks', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('vqa_attention_maps.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_attention_maps(agent, env)

## 8. Question Type vs Attention Pattern Analysis

In [ ]:
def analyze_attention_patterns(agent, env, n_samples=500):
    """Analyze systematic attention patterns for different question types."""
    attention_by_type = {'color': [], 'count': [], 'position': []}
    accuracy_by_type = {'color': [], 'count': [], 'position': []}
    
    for _ in range(n_samples):
        state = env.reset()
        attention = agent.get_attention_map(state)
        attention_grid = attention.reshape(env.grid_size, env.grid_size)
        
        action = agent.select_action(state, training=False)
        _, _, _, info = env.step(action)
        
        attention_by_type[info['q_type']].append(attention_grid)
        accuracy_by_type[info['q_type']].append(info['correct'])
    
    # Plot average attention maps per question type
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    for idx, (qt, attn_list) in enumerate(attention_by_type.items()):
        if attn_list:
            avg_attn = np.mean(attn_list, axis=0)
            im = axes[idx].imshow(avg_attn, cmap='YlOrRd', vmin=0)
            plt.colorbar(im, ax=axes[idx], fraction=0.046)
            
            acc = np.mean(accuracy_by_type[qt])
            axes[idx].set_title(f'{qt.capitalize()} Questions\n'
                               f'(n={len(attn_list)}, acc={acc:.1%})')
            
            # Annotate values
            for i in range(env.grid_size):
                for j in range(env.grid_size):
                    val = avg_attn[i, j]
                    axes[idx].text(j, i, f'{val:.3f}', ha='center', va='center',
                                  fontsize=8, color='black' if val < 0.1 else 'white')
            axes[idx].axis('off')
    
    plt.suptitle('Average Attention Pattern by Question Type', fontsize=13)
    plt.tight_layout()
    plt.savefig('attention_patterns.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    # Entropy analysis
    print("\nAttention Entropy by Question Type:")
    print("(Lower entropy = more focused attention)")
    print("-" * 40)
    for qt, attn_list in attention_by_type.items():
        if attn_list:
            entropies = []
            for attn in attn_list:
                attn_flat = attn.flatten()
                attn_flat = attn_flat / attn_flat.sum()
                entropy = -np.sum(attn_flat * np.log(attn_flat + 1e-10))
                entropies.append(entropy)
            print(f"  {qt:10s}: H = {np.mean(entropies):.3f} ± {np.std(entropies):.3f}")

analyze_attention_patterns(agent, env)

## 9. Question-Conditioned Attention Comparison

In [ ]:
def compare_attention_same_image(agent, env):
    """Show how attention changes for different questions on the same image."""
    # Generate a scene
    state = env.reset()
    base_image = env.image.copy()
    scene_objects = env.scene_objects.copy()
    region_features = env.region_features.copy()
    
    # Generate multiple questions for the same scene
    questions = []
    for pos in ['left', 'right', 'top', 'bottom']:
        q = f"What color is the object on the {pos}?"
        questions.append(q)
    
    for shape in ['circles', 'squares']:
        q = f"How many {shape} are there?"
        questions.append(q)
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes_flat = axes.flatten()
    
    for idx, question in enumerate(questions[:6]):
        # Encode question
        q_encoding = env._encode_question(question)
        
        # Build state with same image features but different question
        state = np.concatenate([region_features.flatten(), q_encoding])
        
        # Get attention
        attention = agent.get_attention_map(state)
        attention_grid = attention.reshape(env.grid_size, env.grid_size)
        
        # Visualize
        ax = axes_flat[idx]
        ax.imshow(base_image)
        attn_up = np.kron(attention_grid, np.ones((env.cell_size, env.cell_size)))
        ax.imshow(attn_up, cmap='hot', alpha=0.5, vmin=0)
        
        # Mark highest attention
        max_idx = attention.argmax()
        max_row = max_idx // env.grid_size
        max_col = max_idx % env.grid_size
        rect = patches.Rectangle(
            (max_col * env.cell_size, max_row * env.cell_size),
            env.cell_size, env.cell_size,
            linewidth=2, edgecolor='cyan', facecolor='none'
        )
        ax.add_patch(rect)
        
        ax.set_title(f'Q: {question}', fontsize=8, wrap=True)
        ax.axis('off')
    
    plt.suptitle('Same Image, Different Questions → Different Attention', fontsize=12)
    plt.tight_layout()
    plt.savefig('question_conditioned_attention.png', dpi=100, bbox_inches='tight')
    plt.show()

compare_attention_same_image(agent, env)

## Summary

### Key Takeaways

1. **Attention as Action:** In VQA, selecting which image region to attend to can be modeled as an RL action. The agent learns that different questions require attending to different regions.

2. **Question-Conditioned Policy:** The policy $\pi(a|s)$ is conditioned on both image features and the question embedding, enabling the same model to handle diverse question types.

3. **Hard vs. Soft Attention:** Our RL approach implements hard attention (selecting one region). The Q-values can be interpreted as soft attention weights showing the agent's uncertainty:
   $$\alpha_i = \frac{\exp(Q(s, a_i) / \tau)}{\sum_j \exp(Q(s, a_j) / \tau)}$$

4. **Specialization by Question Type:** The agent develops distinct attention patterns for different question types (color → specific object, count → distributed, position → spatially biased).

### Formal Framework

The VQA attention problem optimizes:

$$\max_\theta \mathbb{E}_{(\mathbf{I}, q, y^*) \sim \mathcal{D}} \left[ \mathbb{1}\left[ g(\mathbf{f}_{\pi_\theta(s)}, \mathbf{q}) = y^* \right] \right]$$

where:
- $\pi_\theta$ is the attention policy
- $g$ is the answer predictor
- $\mathbf{f}_{\pi_\theta(s)}$ are features of the attended region

This is non-differentiable due to hard attention (discrete action), motivating the RL approach over standard backpropagation.

### Connection to Modern VQA

- **Bottom-Up/Top-Down Attention:** Learned attention over object proposals
- **Transformer Attention:** Self-attention as a differentiable soft version
- **Multi-hop Reasoning:** Multiple attention steps for complex questions
- **MDETR:** End-to-end modulated detection for text-conditioned visual grounding

### Next Steps
- Multi-step attention (attend → reason → attend again)
- Learned question encoder (LSTM/Transformer instead of BoW)
- More complex visual scenes with natural images
- Compositional questions requiring multi-region attention